# Lesson 23: Time Series Forecasting with Support Vector Regressor (SVR)

Reshape hourly electricity load into `[batch, 5]` sliding-window tensors, fit `SVR(kernel='rbf', gamma=0.5, C=10, epsilon=0.05)`, and evaluate train and test MAPE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler

rng = np.random.default_rng(23)
idx = pd.date_range('2014-11-01', '2014-12-31 23:00:00', freq='h')
load = 3100 + 500 * np.sin(2 * np.pi * (idx.hour - 7) / 24.0) + rng.normal(0, 35, len(idx))
energy = pd.DataFrame({'load': load}, index=idx)

train = energy.loc[:'2014-12-29 23:00:00'].copy()
test = energy.loc['2014-12-30 00:00:00':].copy()

scaler = MinMaxScaler()
train_s = scaler.fit_transform(train[['load']])
test_s = scaler.transform(test[['load']])

timesteps = 5
def make_windows(arr):
    mat = np.array([[arr[i+j, 0] for j in range(timesteps)] for i in range(len(arr) - timesteps + 1)])
    return mat[:, :timesteps - 1], mat[:, [timesteps - 1]]

x_train, y_train = make_windows(train_s)
x_test, y_test = make_windows(test_s)

model = SVR(kernel='rbf', gamma=0.5, C=10, epsilon=0.05)
model.fit(x_train, y_train[:, 0])

y_test_pred = scaler.inverse_transform(model.predict(x_test).reshape(-1, 1))
y_test_act = scaler.inverse_transform(y_test)
mape_test = np.mean(np.abs(y_test_act - y_test_pred) / y_test_act) * 100
print(f"Test MAPE: {mape_test:.2f}%")

plt.figure(figsize=(10, 4))
plt.plot(y_test_act, color='red', label='Actual')
plt.plot(y_test_pred, color='blue', label='SVR Predicted')
plt.legend()
plt.title('SVR Time Series Forecast')
plt.show()

## Assignment: A New SVR Model

Experiment with `timesteps=12` and `kernel='poly'` vs `kernel='rbf'` and compare test MAPE.